# Phase 4 detection, Rung 1: ketos segtrain fine-tuning on Kaggle GPU

Fine-tunes kraken's own bundled generic segmentation model (`blla.mlmodel`) on a work-stratified
sample of NomNaOCR's real `gts/*.txt` column ground truth (1,191 pages across all 9 works, 150/
work cap - see `experiments/phase4_detection/README.md`'s Rung 1 section), converted to PageXML
by `scripts/build_segtrain_data.py`.

Escalating to this from a local CPU sanity trial that confirmed fine-tuning-from-bundled produces
a real, improving signal (`val_mean_iu` 0.003->0.032->0.08 over 3 epochs) on 34 pages from a
single work - unlike training from random init, which stayed at 0.0. That trial did not show
whether the approach generalizes across works with different scan characteristics, which is the
actual problem Rung 0 (kraken's generic segmenter + x-position merge, no fine-tuning) failed at:
it passed on its DVSKTT tuning page but scored exactly 0.000/0.000 on 4 of 15 sampled pages from
other works. This run's stratified training set is meant to fix that.

Reuses the environment setup already fully debugged in Phase 0's own CHAT fine-tuning Kaggle run
(`experiments/phase0_validation/chat_finetune_trial.ipynb`, `results.md`): kraken==4.3.13 does not
build on Python 3.12 (Kaggle's default), so this installs a self-contained Miniconda + Python 3.10
env under `/opt` (not `/kaggle/working`, which Kaggle treats as kernel output and would otherwise
re-upload/re-download the whole toolchain on every fetch), plus two already-identified pin fixes
(`setuptools<81` for a dropped `pkg_resources` namespace pattern, `rich<13.4` for a
`pytorch_lightning`/`rich` `Console.clear_live()` incompatibility - note this is a tighter pin than
the `rich<14` that was sufficient locally; kraken's Kaggle-resolved dependency set differs enough
that the wider local pin isn't assumed to carry over unchecked).

In [ ]:
# kraken 4.3.13 needs Python <=3.11; Kaggle's system Python is 3.12, and Kaggle's image has no
# conda on PATH. Install a self-contained Miniconda + Python 3.10 env under /opt (NOT
# /kaggle/working, which Kaggle treats as kernel output).
!curl -sL https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -o /tmp/miniconda.sh
!bash /tmp/miniconda.sh -b -p /opt/miniconda
CONDA = '/opt/miniconda/bin/conda'
# Newer conda requires explicitly accepting the default channels' Terms of Service before
# `conda create` will use them.
!{CONDA} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!{CONDA} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!{CONDA} create -y -n kraken_env python=3.10 -q
!{CONDA} run -n kraken_env pip install --quiet kraken==4.3.13
# kraken pulls in an old pytorch_lightning/lightning_fabric that still does
# pkg_resources.declare_namespace(...); recent setuptools (>=81) dropped that module entirely.
!{CONDA} run -n kraken_env pip install --quiet "setuptools<81"
# pytorch_lightning's RichProgressBar breaks against rich>=13.4 (Console.clear_live() assumes a
# Live was already pushed) - same root cause as the local rich<14 fix, pinned tighter here since
# this is Kaggle's own dependency resolution, not re-assumed from the local result.
!{CONDA} run -n kraken_env pip install --quiet "rich<13.4"
!{CONDA} run -n kraken_env python -c "import torch; print('torch:', torch.__version__); print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"

In [ ]:
import os, glob, shutil, time

def find_dataset_dir(root='/kaggle/input'):
    for dirpath, dirnames, filenames in os.walk(root):
        if any(f.endswith('.xml') for f in filenames):
            return dirpath
    return None

REMOTE_DATASET_DIR = find_dataset_dir()
print('resolved REMOTE_DATASET_DIR:', REMOTE_DATASET_DIR)
if REMOTE_DATASET_DIR is None:
    print('no .xml files found anywhere under /kaggle/input - full tree:')
    for dirpath, dirnames, filenames in os.walk('/kaggle/input'):
        print(dirpath, '->', filenames[:10])
    raise FileNotFoundError('training PageXML files not found under /kaggle/input')

# Copies the dataset off /kaggle/input (commonly FUSE/network-mounted - real per-file-open
# latency was measured here, ~8s/page, before this fix) onto local disk first. Confirmed fast
# for the full 1191-page set (~10-15s).
DATASET_DIR = '/tmp/segtrain_local_data'
if not os.path.isdir(DATASET_DIR):
    t0 = time.time()
    shutil.copytree(REMOTE_DATASET_DIR, DATASET_DIR)
    print(f'copied dataset to local disk in {time.time() - t0:.1f}s')

xml_files = sorted(glob.glob(f'{DATASET_DIR}/*.xml'))
print('training pages found:', len(xml_files))

# RESOLVED: earlier full-scale (1191) and medium-scale (300) attempts appeared to hang for
# 35-65+ minutes with zero log output via Jupyter's `!{cmd}` capture - this was a pure
# monitoring blind spot, not a real stall. A 300-page run using the next cell's explicit
# subprocess+polling visibility fix showed real, steady per-epoch checkpoint progress
# (~90s/epoch) the whole time; `!{cmd}`'s output just never surfaced through Kaggle's log
# capture. Full training set now that both the fine-tuning approach (34-page trial:
# val_mean_iu=0.231 by epoch 14) and the monitoring approach are confirmed working.
XML_LIMIT = None
if XML_LIMIT is not None:
    xml_files = xml_files[:XML_LIMIT]
print('training pages used this run:', len(xml_files))

In [ ]:
import subprocess

CONDA = '/opt/miniconda/bin/conda'
# kraken bundles its own generic segmentation model as package data - no separate weights file
# needs to be uploaded (unlike CHAT's chat_rec.mlmodel in the Phase 0 trial, which isn't bundled
# with kraken itself).
result = subprocess.run(
    [CONDA, 'run', '-n', 'kraken_env', 'python', '-c',
     "import pkg_resources; print(pkg_resources.resource_filename('kraken', 'blla.mlmodel'))"],
    capture_output=True, text=True, check=True,
)
BLLA_MODEL = result.stdout.strip().splitlines()[-1]
print('base segmentation model:', BLLA_MODEL)
assert os.path.exists(BLLA_MODEL), f'expected bundled model at {BLLA_MODEL}'

In [ ]:
# Fine-tune from kraken's bundled weights (confirmed necessary locally - training a fresh net
# from random init on far less data stayed at val_mean_iu=0.0, see README.md). --suppress-regions:
# only baseline/line detection is needed, not region-type classification (a single whole-page
# TextRegion is all build_segtrain_data.py writes). -q early: let the trainer decide when it's
# converged rather than guessing a fixed epoch count - the 34-page GPU diagnostic run was still
# climbing at epoch 14 (0.231) before early-stopping at 24, so a fixed small N would likely stop
# too early on this much larger, more diverse training set. -N 50 stays as the (default) hard cap.
#
# --workers 0: kept from the (mistaken, but harmless) fork+CUDA hypothesis tried earlier - the
# actual root cause of THAT hang turned out to be slow per-file I/O against the remote-mounted
# dataset (see the previous cell), not DataLoader workers.
#
# VISIBILITY FIX: earlier attempts at 300 and 1191 pages ran via `!{cmd}` (Jupyter's own shell
# capture) and showed ZERO new log output for 35-65+ minutes even after the I/O fix - both in
# `kaggle kernels logs -f` and on the Kaggle web UI itself, while the 34-page run's own `!{cmd}`
# output DID eventually appear, just as one large delayed burst. Whether that was real training
# progress hidden behind buffering, or an actual hang, was never resolved - `kaggle kernels files`
# (which should show a `.mlmodel` per completed epoch) was empty in both cases, but that endpoint
# may not reflect a running kernel's live state at all, so it wasn't reliable evidence either way.
# Instead of relying on however Jupyter/Kaggle's log-capture flushes `!{cmd}`'s inherited stdout,
# run segtrain as a background subprocess writing to its own file, and poll+print that file's
# contents explicitly (with real, controlled print() calls) - decoupled from whatever caused the
# earlier silence, and independently verifiable via `kaggle kernels files` showing checkpoint
# `.mlmodel` growth over time in this notebook's own /kaggle/working output regardless.
import subprocess, time, os

DEVICE = 'cuda:0'
OUTPUT_PREFIX = '/kaggle/working/segtrain_finetuned'
TRAIN_LOG = '/kaggle/working/segtrain_train.log'

cmd = [
    CONDA, 'run', '-n', 'kraken_env', 'ketos', 'segtrain', '-f', 'page',
    '-i', BLLA_MODEL, '--resize', 'both', '--suppress-regions', '-d', DEVICE,
    '-N', '50', '-q', 'early', '--min-epochs', '5', '--lag', '10', '-p', '0.9',
    '--workers', '0', '-o', OUTPUT_PREFIX,
] + xml_files
print('launching:', ' '.join(cmd[:12]), '... (+%d xml files)' % len(xml_files))

with open(TRAIN_LOG, 'w', encoding='utf-8') as log_f:
    proc = subprocess.Popen(cmd, stdout=log_f, stderr=subprocess.STDOUT)

start = time.time()
last_pos = 0
while proc.poll() is None:
    time.sleep(30)
    elapsed = time.time() - start
    with open(TRAIN_LOG, encoding='utf-8', errors='replace') as f:
        f.seek(last_pos)
        new_content = f.read()
        last_pos = f.tell()
    n_ckpt = len(glob.glob(f'{OUTPUT_PREFIX}*.mlmodel'))
    print(f'--- [{elapsed:.0f}s elapsed, {n_ckpt} checkpoints so far] ---', flush=True)
    if new_content.strip():
        print(new_content, flush=True)

print(f'process exited with code {proc.returncode} after {time.time() - start:.0f}s total', flush=True)
with open(TRAIN_LOG, encoding='utf-8', errors='replace') as f:
    f.seek(last_pos)
    print(f.read(), flush=True)

In [ ]:
import glob, os
checkpoints = sorted(glob.glob('/kaggle/working/segtrain_finetuned*.mlmodel'))
for c in checkpoints:
    print(c, os.path.getsize(c), 'bytes')